# Baseline Modeling

Compare a majority-class dummy classifier with logistic regression. Evaluate both on the held-out test set using metrics that account for the churn-class imbalance.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

PROJECT_ROOT = Path().resolve().parents[0]
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
TARGET = "Encoded Churn Label"

train_df = pd.read_csv(INTERIM_DIR / "telco_train_features.csv")
test_df = pd.read_csv(INTERIM_DIR / "telco_test_features.csv")

if TARGET not in train_df.columns or TARGET not in test_df.columns:
    raise ValueError(f"Both feature files must contain {TARGET!r}.")

X_train = train_df.drop(columns=TARGET)
y_train = train_df[TARGET].astype(int)
X_test = test_df.drop(columns=TARGET)
y_test = test_df[TARGET].astype(int)

if not X_train.columns.equals(X_test.columns):
    raise ValueError("Training and test feature columns do not match.")

print(f"Training set: {X_train.shape}; test set: {X_test.shape}")
print("Test target counts:")
display(y_test.value_counts().sort_index().rename(index={0: "Retained", 1: "Churned"}))

Training set: (5625, 54); test set: (1407, 54)
Test target counts:


Encoded Churn Label
Retained    1033
Churned      374
Name: count, dtype: int64

In [2]:
models = {
    "Majority-class baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(
        max_iter=1000,
        random_state=123,
        class_weight="balanced",
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    fitted_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced accuracy": balanced_accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(3))

,Accuracy,Balanced accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,,
Majority-class baseline,0.734,0.500,0.000,0.000,0.000,0.500
Logistic regression,0.768,0.779,0.543,0.802,0.648,0.854


In [3]:
for name, model in fitted_models.items():
    predictions = model.predict(X_test)

    print(f"\n{name}")
    print("Confusion matrix (rows = actual, columns = predicted):")
    display(pd.DataFrame(
        confusion_matrix(y_test, predictions),
        index=["Actual retained", "Actual churned"],
        columns=["Predicted retained", "Predicted churned"],
    ))
    print(classification_report(
        y_test,
        predictions,
        target_names=["Retained", "Churned"],
        zero_division=0,
    ))


Majority-class baseline
Confusion matrix (rows = actual, columns = predicted):


,Predicted retained,Predicted churned
Actual retained,1033,0
Actual churned,374,0


              precision    recall  f1-score   support

    Retained       0.73      1.00      0.85      1033
     Churned       0.00      0.00      0.00       374

    accuracy                           0.73      1407
   macro avg       0.37      0.50      0.42      1407
weighted avg       0.54      0.73      0.62      1407


Logistic regression
Confusion matrix (rows = actual, columns = predicted):


,Predicted retained,Predicted churned
Actual retained,781,252
Actual churned,74,300


              precision    recall  f1-score   support

    Retained       0.91      0.76      0.83      1033
     Churned       0.54      0.80      0.65       374

    accuracy                           0.77      1407
   macro avg       0.73      0.78      0.74      1407
weighted avg       0.82      0.77      0.78      1407

